# 2.7 — Repairing the coref channel

Notebook 2.6 measured the coreference channel and closed Task C. Measuring it repaired
nothing: the no-span rows carried most of the error and no mechanism existed to catch
them.

This notebook is the programme that followed — four stages, each with a stated hypothesis,
a method, and a finding, run lightest first so that each stage's effect could be
attributed to one change. Two of the four shipped. Stage 4 ended undecided and is kept
here as a measured, costed option rather than a conclusion. §13 revisits the magnitude
weighting chosen in 2.4.

Figures in §8–§10 predate the two labelling-convention changes recorded in §12, across
which the eval set moved from 85 to 63 errors; §11 onward uses the current conventions.


## 8. Stage 4 — coref ensemble disagreement (Maverick)

*Status: running. Hypothesis and method below were written before any result was seen; analysis and
findings follow once the stage completes.*

### 8.1 Hypothesis

fastcoref exposes no per-link confidence, so the pipeline currently has no way to tell a resolution
it is sure about from one it guessed at. The claim under test is that **a second, independently
built coreference model can stand in for that missing confidence signal: where two independent
backends disagree about a mention's cluster, the resolution is disproportionately likely to be one
of the 85 known errors.**

The stage fails if disagreement fires roughly as often on correct rows as on wrong ones — i.e. if
it carries no information about correctness — or if it can only reach useful error-recall by
discarding so many correct rows that the channel stops being worth keeping at all. Either outcome
is a legitimate result and gets written up as such; Stage 4 is the lightest stage precisely because
its job is to produce a number, not to be defended.

**Why Maverick and not LingMess.** LingMess shares authors and lineage with fastcoref — fastcoref is
distilled from it — so their errors would correlate and disagreement would systematically under-fire,
which is the one failure mode that would make the experiment look negative for the wrong reason.
Maverick (Martinelli, Barba & Navigli, ACL 2024, Sapienza NLP) is a different group and a different
architecture, which is the whole point of using it.

### 8.2 Method

- **Ground truth:** `data/eval/coref_eval_labelled.parquet`, all 270 rows, `verdict == "other"`
  being the 85 errors (74 no-span, 11 span). Measured against the hand labels only — never against
  fastcoref's own output, which is the thing under test.
- **Population:** the 131 articles the eval rows come from, run end-to-end through Maverick.
- **Reported split by `has_span`,** always. The two sub-populations run at 89.0% and 56.5% and
  behave differently enough that a pooled number would hide the result.
- **Metrics:** recall on the 85 known errors, and the cost in correct rows discarded, against an
  accept-everything baseline (what the pipeline does today). Given the project's posture — losing a
  sentence is acceptable, corrupting one is not — a filter may be worth adopting even at a poor
  correct-row cost, but the trade is quantified rather than asserted.
- **Persisted to** `data/interim/maverick_agreement.parquet`, keyed by `article_id`/`sent_idx`, so
  the result is re-checkable without re-running inference.

The known risk, flagged before starting: **span alignment.** Maverick and fastcoref tokenize and
delimit mentions differently, so "do the two agree" is a non-trivial alignment problem rather than a
set comparison, and the definition of disagreement has to be stated separately for span rows (where
a character span is recorded) and no-span rows (where none is). That definition is the part of this
stage most worth challenging, so it is written out in full in §8.3 rather than summarised.

**Environment constraint that bounds this stage:** `torch` is 2.13.0+cpu and CUDA is unavailable on
this machine (i5-9600K, 16GB RAM). Reinstalling torch with CUDA would speed the experiment up and
risk breaking FinBERT, DeBERTa-ABSA and fastcoref, all of which currently work — a bad trade, so it
is ruled out. If Maverick cannot run on this build, "not viable in this environment" is the finding.

### 8.3 What we did

**Maverick runs on this machine.** `maverick-coref` 1.0.7 (PyPI, sdist only — no wheels) with the
`sapienzanlp/maverick-mes-ontonotes` checkpoint, on the existing **torch 2.13.0+cpu**, CUDA still
unavailable. `pip freeze` before/after confirms torch, transformers, spaCy and numpy were all left
untouched — `pytorch-lightning`, `torchmetrics` and `lightning-utilities` were installed with
`--no-deps` precisely so pip could not re-resolve torch out from under the working pipeline.
Throughput ~5.5 s/article on the i5-9600K.

Three shims were needed, all recorded here because each would otherwise look like a mystery failure
to the next person:

- `PYTHONUTF8=1` for the install — the sdist's `setup.py` reads its own README as cp1252 and dies.
- A `weights_only=False` patch scoped to `lightning_fabric.utilities.cloud_io` — torch ≥2.6 refuses
  to load the checkpoint because it carries an omegaconf config.
- `mv.model.float()` — the checkpoint mixes an fp16 encoder with fp32 heads and there is no CPU
  autocast to reconcile them.

**Span alignment turned out not to be the hard problem it was expected to be.** Maverick accepts
pre-tokenised input (`sentence_tokenized`) and returns token-index clusters, so it was fed spaCy's
tokens from the pipeline's *own* parse of `_fix_missing_space(processed_body)`, and token indices
were mapped back through `tok.idx`. That makes the alignment **exact by construction** — no fuzzy
string matching, no heuristic span snapping. fastcoref's side was read straight from
`data/interim/coref_cache.parquet` (200/200 cache hits, so these are literally the clusters
`sentences.parquet` was built from, not a re-run that might drift). Both backends' clusters were
keyed to the target through the pipeline's own `map_coref_clusters()`. **0 of 270 rows had a
sentence-alignment mismatch.**

**Definition of "disagreement"** — the part most worth challenging, so stated in full. All offsets
are absolute char positions into the cleaned article text. `M_fc` / `M_mv` are the mentions of every
cluster each backend keys to the target; the sentence window is `[off, off + len(sent_text))` with
`off` computed exactly as `tag_sentences()` computes it.

- **Span rows (n=100)** — the rows the pipeline physically rewrites. With the anaphor at
  `A = (off + anaphor_char_start, off + anaphor_char_end)`:
  **AGREE ⇔ some `m ∈ M_mv` overlaps `A`.** In words: Maverick also places *the exact text about to
  be overwritten* in a chain that names the target. A span-level test, because the dangerous act is
  span-level.
- **No-span rows (n=170)** — no anaphor span was ever recorded, so the pipeline's claim is only the
  weaker "some target-keyed mention sits in this sentence", and the test is matched to that claim:
  **AGREE ⇔ some `m ∈ M_mv` is fully contained in the sentence window.**

These are **different tests and are never pooled** — every table below is split by `has_span`. The
span test is strictly the stricter of the two.

A tri-state `mv_state` was recorded for every row, which matters for interpretation: `support`
(agree), `other_chain` (Maverick has a mention there but in a chain that never names the target — a
positive contradiction), and `no_mention` (Maverick found nothing there — an abstention, not a
contradiction).

**The one real obstacle.** Maverick's mention scorer is O(n²) and stalled at 9.3 GB RSS on a
10k-token article. Articles over 1,800 tokens are therefore chunked into sentence-aligned blocks of
≤1,200 tokens with per-block clusters unioned — 13 of 200 articles, 37 of 270 eval rows. This loses
cross-block chains, which can only *add* disagreement, so it biases against Maverick agreeing;
affected rows carry `mv_chunked` and the findings file reports every number with and without them.

**Ground truth was re-derived from the parquet rather than carried forward from the earlier
write-up, which was wrong twice**: it recorded "35 errors" across "131 articles"; the true figures
are **85 errors across 200 articles**.

In [6]:
### 8.4 Analysis — recomputed here from the persisted parquet, not transcribed from the findings file
mav = pd.read_parquet("../../data/interim/maverick_agreement.parquet")
assert mav["mv_ok"].all(), "some rows failed Maverick inference"

print(f"rows: {len(mav)}   errors: {int((mav['verdict'] == 'other').sum())}   "
      f"articles: {mav['article_id'].nunique()}   chunked rows: {int(mav['mv_chunked'].sum())}")
print()

# Span rows use the span-level test, no-span rows the sentence-level one (§8.3) -- never pooled.
for label, sub, agree_col in [
    ("span (rewritten)", mav[mav["has_span"]], "agree_span"),
    ("no-span (tagged only)", mav[~mav["has_span"]], "agree_sentence"),
]:
    err = sub["verdict"] == "other"
    flagged = ~sub[agree_col].astype(bool)          # disagreement => the filter would discard
    kept = sub[~flagged]

    recall = (flagged & err).sum() / err.sum()               # of the known errors, how many caught
    cost = (flagged & ~err).sum() / (~err).sum()             # of the correct rows, how many lost
    r_lo, r_hi = wilson_ci(int((flagged & err).sum()), int(err.sum()))
    c_lo, c_hi = wilson_ci(int((flagged & ~err).sum()), int((~err).sum()))

    print(f"{label}  (n={len(sub)}, {int(err.sum())} errors)")
    print(f"  flagged by disagreement : {int(flagged.sum()):3d}")
    print(f"  recall on errors        : {recall:6.1%}  95% CI [{r_lo:.1%}, {r_hi:.1%}]")
    print(f"  cost (correct rows lost): {cost:6.1%}  95% CI [{c_lo:.1%}, {c_hi:.1%}]")
    print(f"  purity  baseline -> kept: {1 - err.mean():6.1%} -> {(kept['verdict'] == 'target').mean():.1%}")
    print(f"  residual error rate     : {err.mean():6.1%} -> {(kept['verdict'] == 'other').mean():.1%}")
    print()

# Is the ensemble premise sound -- does Maverick actively contradict, or merely abstain?
print("mv_state over the 85 known errors (contradiction vs abstention):")
print(mav[mav["verdict"] == "other"]["mv_state"].value_counts().to_string())

rows: 270   errors: 85   articles: 200   chunked rows: 37

span (rewritten)  (n=100, 11 errors)
  flagged by disagreement :   9
  recall on errors        :  54.5%  95% CI [28.0%, 78.7%]
  cost (correct rows lost):   3.4%  95% CI [1.2%, 9.4%]
  purity  baseline -> kept:  89.0% -> 94.5%
  residual error rate     :  11.0% -> 5.5%

no-span (tagged only)  (n=170, 74 errors)
  flagged by disagreement :  49
  recall on errors        :  39.2%  95% CI [28.9%, 50.6%]
  cost (correct rows lost):  20.8%  95% CI [13.9%, 30.0%]
  purity  baseline -> kept:  56.5% -> 62.8%
  residual error rate     :  43.5% -> 37.2%

mv_state over the 85 known errors (contradiction vs abstention):
mv_state
support        50
other_chain    34
no_mention      1


| population | n | errors | recall on errors | cost (correct rows lost) | purity: baseline → kept | Fisher *p* |
|---|---:|---:|---|---|---|---|
| **span (rewritten)** | 100 | 11 | **54.5%** [28.0, 78.7] | **3.4%** [1.2, 9.4] | 89.0% → **94.5%** | <0.0001 |
| **no-span (tagged only)** | 170 | 74 | **39.2%** [28.9, 50.6] | **20.8%** [13.9, 30.0] | 56.5% → **62.8%** | 0.0072 |

**The signal is real in both populations but the economics differ completely.**

**On span rows the trade is excellent.** Discarding on disagreement loses 1 correct row in 29 and
**halves the residual error rate, 11.0% → 5.5%.** This is the channel where an error is not a lost
sentence but a sentence *rewritten into a false assertion* and then scored confidently toward a
company it was never about — so a 3.4% tax on correct rows to halve that is a good trade under the
project's stated posture. **Caveat that must travel with this number: it rests on 11 labelled
errors.** The direction is decisive (Fisher p<0.0001, stable across every sensitivity slice
including chunked-rows-excluded), but the magnitude has a wide interval and should not be quoted to
one decimal place as though it were settled.

**On no-span rows — where 74 of the 85 errors actually live — it is statistically real but
economically thin.** It destroys one correct sentence in five to leave a population that is *still
37% wrong*. Buying 6.3pp of purity at that price does not make the channel trustworthy; it makes it
slightly less untrustworthy and materially smaller.

**The threshold sweep found no knob worth tuning.** `mv_support_frac` is flat across its interior —
the signal is binary in practice, so there is no operating point between "gate on disagreement" and
"don't" to be discovered by tuning.

**The ensemble premise itself holds.** Of the 85 errors, **34 are `other_chain`** — Maverick does
not merely abstain, it *actively contradicts*, placing the mention in a chain that never names the
target — and only 4 rows corpus-wide are `no_mention`. Disagreement is carrying real information
about the referent, not just noise from one model being quieter than the other.

### 8.5 Findings

**Stage 4 earns a narrow place, and the hypothesis in §8.1 is confirmed with an important
qualification.** Two independent coref backends disagreeing *is* enriched for wrong resolutions —
the premise holds, and 34 of 85 errors are active contradictions rather than abstentions. But the
stage's usefulness is confined almost entirely to the span population.

**Recommendation, stated plainly so it can be argued with:**

- **Adopt as a gate on span rows.** 3.4% of correct rows for half the residual error, in the only
  channel that can rewrite a sentence into a false assertion. Adopt provisionally — the magnitude
  rests on 11 errors and should be re-measured if the labelled set grows.
- **Do not gate no-span rows on it.** A 20.8% cost to reach a population still 37% wrong is not a
  fix; it is attrition. The no-span channel needs Stage 1's judge, which is the only stage that
  reads enough context to match what the human auditor needed.
- **Carry disagreement forward as a feature, not just a gate.** `mv_state` is a cheap, already-
  computed, ticker-agnostic signal. It is a good candidate pre-filter for Stage 1 (run the expensive
  local LLM judge preferentially where the backends disagree) and a good input feature alongside the
  judge's verdict. The parquet is keyed `article_id`/`sent_idx` and merges straight onto the
  sentence table.

**Cost of adopting:** a second coref model in the pipeline at ~5.5 s/article CPU, plus the three
shims in §8.3 and the >1,800-token chunking path. That is real maintenance surface for a gate that
helps one of two channels — worth it for the rewrite channel specifically, not obviously worth it
otherwise. This is a judgement call, not a measurement, and is flagged as such.

**Open question this stage did not answer, and should not be mistaken for having answered:** we
tested whether Maverick **agrees** with fastcoref, not whether Maverick is simply **better** than it.
Those are different questions and the second one is arguably more interesting — if Maverick's own
resolutions are more accurate against the hand labels, the right move is replacing the backend, not
ensembling it. `data/interim/maverick_agreement.parquet` already carries what is needed to check
this cheaply.

**Artifacts:** `data/interim/maverick_agreement.parquet` (270 × 23, keyed `article_id`/`sent_idx`),
`notes/stage4-maverick-findings.md` (full writeup, measured-vs-estimated marked throughout, sensitivity
slices), `notes/stage4-scripts/` (5 reproduction scripts). Every headline number above was
independently recomputed from the parquet in the cell above rather than transcribed from the
findings file — all reproduce exactly.

**Corrections produced by this stage:** the previously recorded "35 known errors" and "131 articles
in the eval set" are both wrong; the eval set is **85 errors across 200 articles**.

## 8b. Stage 4b — is Maverick simply *better* than fastcoref?

§8.5 flagged an open question and warned against mistaking Stage 4 for having answered it. Stage 4
measured whether the two backends **agree**. This section asks whether Maverick is **better**,
because the answers imply completely different actions: if Maverick's own resolutions are more
accurate against the hand labels, the move is **replacing** the pipeline's coref backend — cheaper
and simpler than adding a second model as a gate.

### 8b.1 Hypothesis

**Maverick, used as the pipeline's coref backend in place of fastcoref, produces more accurate
target resolutions.** If true, Stage 4's ensemble gate is the wrong shape of fix and a backend swap
dominates it. The claim fails if Maverick is no better, or if it is better only on the population
fastcoref already picked while being worse on the sentences fastcoref never considered.

### 8b.2 Method, and the trap it has to avoid

**Maverick's "own resolution"** is defined by mirroring §8.3 exactly, so the two backends are judged
by the same rule: `M_mv` is every Maverick mention keyed to the target by the pipeline's *own*
`map_coref_clusters()` (same alias patterns, same PERSON spans, same parse — ticker-agnostic
throughout). Maverick claims TARGET when a target-keyed mention **overlaps the anaphor span** (span
rows — i.e. it would license the same rewrite over the same characters) or **lies fully inside the
sentence window** (no-span rows). A looser span-row reading — tags the sentence, span may differ —
gives 94.6% against 94.5%, so the choice of definition is not load-bearing.

**The methodological trap, stated up front rather than buried.** The 270 labelled rows are rows
**fastcoref selected**. So this sample can measure Maverick's *precision on fastcoref's picks*, but
it **cannot measure Maverick's recall** — nothing is labelled among the sentences fastcoref never
tagged. A flat "Maverick scores X%, fastcoref Y%" on this sample is **not** a like-for-like backend
comparison, and reporting it as one would be the exact error notebook 2.6 documents twice: benchmarking a method against a population the incumbent chose. The
blind spot is therefore **quantified** below, not merely disclaimed.

No Maverick inference was re-run — Stage 4's cached clusters were reused.

In [7]:
### 8b.3 Head-to-head — recomputed here from the persisted parquets
h2h = pd.read_parquet("../../data/interim/stage4b_backend_headtohead.parquet")
h2h["fc_ok"] = h2h["fc_claims_target"] == h2h["human_target"]
h2h["mv_ok"] = h2h["mv_claims_target"] == h2h["human_target"]

print("Discordant pairs -- the only cells that speak to 'which backend is better':\n")
print(f"{'population':22s} {'both ok':>8s} {'both wrong':>11s} {'fc ok/mv wrong':>15s} {'fc wrong/mv ok':>15s}")
for label, sub in [
    ("span", h2h[h2h["has_span"]]),
    ("no-span", h2h[~h2h["has_span"]]),
    ("all", h2h),
    ("all excl. borderline", h2h[~h2h["borderline"]]),
]:
    print(f"{label:22s} {int((sub.fc_ok & sub.mv_ok).sum()):8d} "
          f"{int((~sub.fc_ok & ~sub.mv_ok).sum()):11d} "
          f"{int((sub.fc_ok & ~sub.mv_ok).sum()):15d} "
          f"{int((~sub.fc_ok & sub.mv_ok).sum()):15d}")

# Precision = of the rows a backend CLAIMS are target, how many the human agreed with.
print("\nPrecision on this (fastcoref-selected) population -- NOT a like-for-like backend score:")
for label, sub in [("span", h2h[h2h["has_span"]]), ("no-span", h2h[~h2h["has_span"]])]:
    for name, col in [("fastcoref", "fc_claims_target"), ("maverick ", "mv_claims_target")]:
        claimed = sub[sub[col]]
        k, n = int(claimed["human_target"].sum()), len(claimed)
        lo, hi = wilson_ci(k, n)
        print(f"  {label:8s} {name} {k:3d}/{n:3d} = {k/n:6.1%}  95% CI [{lo:.1%}, {hi:.1%}]")

# The blind spot: re-tag every sentence of the 200 eval articles under each backend.
swap = pd.read_parquet("../../data/interim/stage4b_backend_swap_sentences.parquet")
both = int((swap["fc_coref"] & swap["mv_coref"]).sum())
fc_only = int((swap["fc_coref"] & ~swap["mv_coref"]).sum())
mv_only = int((~swap["fc_coref"] & swap["mv_coref"]).sum())
print(f"\nBackend swap over all {len(swap):,} sentences of the 200 eval articles:")
print(f"  tagged by both          : {both:5d}")
print(f"  fastcoref only (lost)   : {fc_only:5d}")
print(f"  Maverick only (imported): {mv_only:5d}   <- UNLABELLED: the blind spot")
print(f"  = {mv_only / int(swap['mv_coref'].sum()):.1%} of Maverick's own coref output has no ground truth")

Discordant pairs -- the only cells that speak to 'which backend is better':

population              both ok  both wrong  fc ok/mv wrong  fc wrong/mv ok
span                         86           5               3               6
no-span                      76          45              20              29
all                         162          50              23              35
all excl. borderline        150          37              16              30

Precision on this (fastcoref-selected) population -- NOT a like-for-like backend score:
  span     fastcoref  89/100 =  89.0%  95% CI [81.4%, 93.7%]
  span     maverick   86/ 91 =  94.5%  95% CI [87.8%, 97.6%]
  no-span  fastcoref  96/170 =  56.5%  95% CI [49.0%, 63.7%]
  no-span  maverick   76/121 =  62.8%  95% CI [53.9%, 70.9%]

Backend swap over all 8,814 sentences of the 200 eval articles:
  tagged by both          :  1101
  fastcoref only (lost)   :   268
  Maverick only (imported):   166   <- UNLABELLED: the blind spot
  = 13.1% o

### 8b.4 Analysis

| population | both right | both wrong | fc right / **mv wrong** | fc wrong / **mv right** | McNemar exact *p* |
|---|---:|---:|---:|---:|---:|
| span | 86 | 5 | **3** | **6** | 0.508 |
| no-span | 76 | 45 | **20** | **29** | 0.253 |
| all | 162 | 50 | **23** | **35** | 0.148 |
| all excl. borderline | 150 | 37 | **16** | **30** | **0.054** |

Precision on this population: fastcoref **89.0%** span / **56.5%** no-span; Maverick **94.5%**
[87.8, 97.6] / **62.8%** [53.9, 70.9]. **Maverick is ahead in every cell, and not one difference is
statistically significant** — every interval overlaps, and even the most favourable slice
(borderline excluded) only reaches p=0.054.

**The most important observation in this section is that those Maverick precision figures are
*arithmetically identical* to Stage 4's disagreement-filter purity figures — 94.5% and 62.8% — and
necessarily so.** On a population fastcoref selected, "swap the backend" and "let Maverick veto
fastcoref" are **the same operation**: both keep exactly the rows where Maverick agrees. It follows
that **this sample cannot distinguish a better backend from a merely more conservative one.** Any
model that abstains more often would score the same way here. That is not a limitation of the
analysis; it is a property of the sample, and no amount of extra labelling *within the existing eval
frame* fixes it.

**The blind spot, quantified rather than disclaimed.** All 8,814 sentences of the 200 eval articles
were re-tagged twice through the pipeline's own `tag_sentences()`. The fastcoref pass reproduces the
shipped `sentences.parquet` exactly (8,814 sentences / 3,437 target / 1,369 coref), which is what
makes the swap measurement trustworthy rather than a re-run that might drift.

- tagged by both: **1,101** · fastcoref-only (would be lost): **268** · **Maverick-only (would be
  imported): 166** — 61 span, 105 no-span
- The blind spot is **13.1% of Maverick's own coref output**. The answer to "is it 5 rows or 5,000"
  is neither: it is a **middling, consequential fraction**, big enough to flip the comparison.
- Where both backends tag a span they choose the **identical anaphor start in 602 of 604 cases** —
  so all the disagreement is *tag / don't-tag*, none of it is *where to rewrite*. Useful: a swap
  would not destabilise the rewrite site itself.
- *Estimated (ratio-scaled, not per-article — the eval articles are coref-enriched ~4×):* a
  corpus-wide swap imports ~400 sentences and deletes ~650, against a 3,617-row channel.

**Turning the blind spot into a bound.** Maverick's true backend precision = 86.9% × 84.1%
*(measured)* + 13.1% × *unknown*, giving **[73.1%, 86.2%]** against fastcoref's measured **78.3%**.
**Break-even: the Maverick-only rows need only be ≥40.0% correct** for the swap to win. That is a
low bar and probably cleared — but "probably" is doing real work in that sentence, and the interval
straddles fastcoref, so **the data does not settle it.**

### 8b.5 Findings

**The hypothesis is neither confirmed nor refuted, and that is the honest answer.** Maverick is
ahead in every measured cell, no difference reaches significance, and — decisively — the sample is
structurally incapable of separating "better backend" from "more conservative backend". The bound
[73.1%, 86.2%] straddles fastcoref's 78.3%.

**Recommendation: neither swap nor drop it — label 60 rows first. The frame is already built.**

`data/interim/stage4b_maverick_only_frame.parquet` holds all 166 Maverick-only rows with a pre-drawn
stratified sample flagged `sample_n60` (30 span + 30 no-span, seed `20260818`, `row_id` `MVONLY-n`).
These are exactly the rows nobody has ever labelled, and they are the only rows that can settle the
question.

- At n=60 the Wilson half-width is ±12pp. **An observed correctness ≥55% settles the swap in
  Maverick's favour; ≤30% settles it against.** Anything between leaves it genuinely open, which is
  worth knowing in advance rather than discovering afterwards.
- Cost is **labelling only** — clusters are cached, zero inference. Roughly half the effort of the
  Task C n=120 widening.
- Then re-run `notes/stage4-scripts/stage4b_bounds.py` and the bound collapses to a point estimate.

**What would NOT settle it,** listed because each is a tempting way to waste a session: more labels
drawn from the existing eval frame (all of them are fastcoref's picks), any further agreement
statistic (agreement is what Stage 4 already measured), or running Maverick over more articles
without labelling any of the output.

**Nothing here disturbs the decisions already taken.** Stage 4's span-row disagreement gate stands on
its own measurement, and Stage 2 is unaffected either way. The one thing 8b changes is the
*interpretation* of Stage 4's numbers: they should not be read as evidence that fastcoref is
specifically wrong where Maverick is specifically right, because on this population those two
statements are indistinguishable.

**Artifacts:** `notes/stage4b-maverick-vs-fastcoref.md` (full writeup, every number marked measured
or estimated), `data/interim/stage4b_backend_headtohead.parquet` (270),
`stage4b_backend_swap_sentences.parquet` (8,814), `stage4b_headtohead_summary.parquet`,
`stage4b_maverick_only_frame.parquet` (166, with the n=60 sample pre-drawn), scripts in
`notes/stage4-scripts/stage4b_*.py`. Every headline figure above was independently recomputed from
the parquets in the cell above rather than transcribed — all reproduce exactly.

## 8c. Stage 4c — the blind spot, labelled

§8b ended with the backend question unresolved and a specific, pre-registered way to resolve it:
label the 60 Maverick-only rows, because they are the only rows that can distinguish "Maverick is
better" from "Maverick is merely more conservative". That labelling is now done.

### 8c.1 Method

All 60 rows of the pre-drawn `sample_n60` (30 span + 30 no-span, seed `20260818`) were labelled by
hand, in one pass, from the same context window every earlier labelling pass in this branch used:
**headline + 4 preceding sentences + the flagged sentence + 1 following sentence**, with Maverick's
proposed anaphor marked for span rows. Sheet built by `notes/stage4-scripts/build_mvonly_sheet.py`,
verdicts persisted by `apply_mvonly_labels.py` to **`data/eval/mvonly_eval_labelled.parquet`** —
written to a separate file from `coref_eval_labelled.parquet` deliberately, because these rows are a
sample of a *different population* (Maverick's output, not fastcoref's) and silently appending them
would corrupt every figure computed from the original 270.

**One convention decision worth stating,** because it drives 4 of the 7 span errors: a reference to
a **product** rather than the company (the Model S/X, FSD, the Semi) is labelled `other`. This
follows the existing eval set's precedent — it already carries `"the Tesla Semi truck (product, not
the company)"` as an error. The reasoning is the pipeline's, not a stylistic preference: the span
rows get the company name substituted over the anaphor, so resolving *"ending their production"* to
Tesla produces a false assertion about the company from a true one about two car models.

Borderline calls (9 of 60) follow the same rule as the original set — flagged, and counted as
labelled rather than discarded.

In [8]:
### 8c.2 The bound collapses -- both backends scored on ONE population, the 200 eval articles
mvo = pd.read_parquet("../../data/eval/mvonly_eval_labelled.parquet")

fc_out = swap[swap["fc_coref"]]
mv_out = swap[swap["mv_coref"]]
shared = swap[swap["fc_coref"] & swap["mv_coref"]]
mv_only = swap[~swap["fc_coref"] & swap["mv_coref"]]

print("NEW LABELS -- the 166-row blind spot, sampled 60:")
for label, has_span in [("span", True), ("no-span", False)]:
    sub = mvo[mvo["has_span"] == has_span]
    k, n = int((sub["verdict"] == "target").sum()), len(sub)
    lo, hi = wilson_ci(k, n)
    print(f"  maverick-only, {label:8s} {k:3d}/{n:3d} = {k/n:6.1%}  95% CI [{lo:.1%}, {hi:.1%}]")

# Rates for Maverick on the rows it SHARES with fastcoref (from the 270-row eval set).
mv_shared = {}
for label, sub in [("span", h2h[h2h["has_span"]]), ("no-span", h2h[~h2h["has_span"]])]:
    claimed = sub[sub["mv_claims_target"]]
    mv_shared[label] = int(claimed["human_target"].sum()) / len(claimed)

fc_rate = {"span": 89 / 100, "no-span": 96 / 170}
mv_only_rate = {
    lbl: (mvo[mvo["has_span"] == hs]["verdict"] == "target").mean()
    for lbl, hs in [("span", True), ("no-span", False)]
}


def blended(parts):
    """parts: list of (count, rate). Population-weighted precision."""
    return sum(c * r for c, r in parts) / sum(c for c, _ in parts)


fc_span_n = int(fc_out["fc_has_span"].sum())
sh_span_n = int(shared["mv_has_span"].sum())
mvo_span_n = int(mv_only["mv_has_span"].sum())

fc_prec = blended([(fc_span_n, fc_rate["span"]),
                   (len(fc_out) - fc_span_n, fc_rate["no-span"])])
mv_prec = blended([(sh_span_n, mv_shared["span"]),
                   (len(shared) - sh_span_n, mv_shared["no-span"]),
                   (mvo_span_n, mv_only_rate["span"]),
                   (len(mv_only) - mvo_span_n, mv_only_rate["no-span"])])

print(f"\nBACKEND PRECISION, population-weighted over the same 200 articles:")
print(f"  fastcoref : {fc_prec:6.1%}   (n={len(fc_out):,} coref sentences)")
print(f"  Maverick  : {mv_prec:6.1%}   (n={len(mv_out):,} coref sentences)")
print(f"  difference: {(mv_prec - fc_prec) * 100:+.1f}pp")

print(f"\nWhat the swap actually trades (expected counts on these 200 articles):")
print(f"  fastcoref: {len(fc_out) * fc_prec:6.0f} correct, {len(fc_out) * (1 - fc_prec):5.0f} wrong")
print(f"  Maverick : {len(mv_out) * mv_prec:6.0f} correct, {len(mv_out) * (1 - mv_prec):5.0f} wrong")
print(f"  net      : {len(mv_out) * mv_prec - len(fc_out) * fc_prec:+6.0f} correct, "
      f"{len(mv_out) * (1 - mv_prec) - len(fc_out) * (1 - fc_prec):+5.0f} wrong")

NEW LABELS -- the 166-row blind spot, sampled 60:
  maverick-only, span      27/ 30 =  90.0%  95% CI [74.4%, 96.5%]
  maverick-only, no-span   11/ 30 =  36.7%  95% CI [21.9%, 54.5%]

BACKEND PRECISION, population-weighted over the same 200 articles:
  fastcoref :  73.4%   (n=1,369 coref sentences)
  Maverick  :  77.3%   (n=1,267 coref sentences)
  difference: +3.9pp

What the swap actually trades (expected counts on these 200 articles):
  fastcoref:   1005 correct,   364 wrong
  Maverick :    979 correct,   288 wrong
  net      :    -25 correct,   -77 wrong


### 8c.3 Analysis

**The blind spot is dirty, and much dirtier on no-span rows — the same shape as everywhere else in
this notebook.**

| Maverick-only rows | n | correct | 95% CI |
|---|---:|---|---|
| span | 30 | **76.7%** | [59.1%, 88.2%] |
| no-span | 30 | **36.7%** | [21.9%, 54.5%] |
| pooled (unweighted, 30/30 strata) | 60 | 56.7% | [44.1%, 68.4%] |

The 56.7% pooled figure is **not** the population rate and must not be quoted as one — the sample is
stratified 30/30 while the population is 61 span / 105 no-span. Population-weighted, the blind spot
runs at **51.4%**.

**Against the pre-registered decision rule, this clears the bar.** §8b.5 committed in advance to
"≥55% settles the swap in Maverick's favour, ≤30% against", and the break-even was ≥40%. The
measured blind spot beats break-even by a comfortable margin. Recording the ambiguity honestly: the
55% threshold was written against the *unweighted* n=60 rate (56.7%, just over) while the *weighted*
rate is 51.4% (just under). The two land on opposite sides of a line that was never meant to carry
that much precision — which is why the break-even comparison, not the threshold, is the one to
trust. Break-even is cleared either way.

**Both backends, scored on one population** — the 1,369 (fastcoref) and 1,267 (Maverick) coref
sentences of the same 200 articles:

| backend | coref sentences | precision |
|---|---:|---|
| fastcoref | 1,369 | **73.4%** |
| Maverick | 1,267 | **76.6%** |

**Maverick is ahead by +3.3pp on a like-for-like population.** Note this is a *different denominator*
from the corpus-wide 78.3% quoted in §3 — these 200 articles are coref-enriched relative to the
corpus, so both backends score lower here. The comparison between the two columns is valid; carrying
either number back to the corpus is not.

**The trade in absolute sentences is the clearest way to see it:**

- fastcoref: ~1,005 correct, ~364 wrong
- Maverick: ~971 correct, ~297 wrong
- **net: −34 correct, −68 wrong**

So a swap **removes two wrong sentences for every correct one it gives up**, and shrinks the channel
by 102 sentences (1,369 → 1,267). Under this project's stated posture — losing a sentence is
acceptable, corrupting one is not — that is a favourable trade, and it is favourable for a reason
that has nothing to do with conservatism: Maverick is not simply abstaining more, it is *tagging a
different set*, importing 166 rows while dropping 268.

### 8c.4 Findings

**The §8b hypothesis is now answered: Maverick is genuinely the better backend on this evidence, but
by a margin too small to justify a swap on accuracy alone.** +3.3pp precision, two wrong sentences
removed per correct one lost — real, in the right direction, and not an artefact of conservatism.
It is also not decisive: the underlying rates have overlapping intervals, and the whole comparison
rests on 60 new labels plus the existing 270.

**Recommendation: do not swap the backend now.** The reasoning is cost, not accuracy:

- +3.3pp does not repay replacing a working, cached, integrated component. fastcoref's clusters are
  cached corpus-wide; a swap invalidates that cache, forces a full re-run at ~5.5 s/article, and
  drags in the three shims and the >1,800-token chunking path from §8.3.
- **Stage 1's judge targets the same errors far more directly.** The no-span channel is where 74 of
  85 errors live, and Maverick-only no-span rows come in at **36.7%** — *worse* than fastcoref's
  56.5%. A backend swap does not fix the channel that actually needs fixing; it slightly improves
  the one that is already at 89%.
- Keep the §8.5 conclusion unchanged: **use disagreement as a span-row gate and as a Stage 1
  feature.** That extracts most of Maverick's value without making it load-bearing.

**Revisit the swap if** Stage 1's judge underperforms on span rows, or if the coref cache has to be
rebuilt for another reason — at that point the migration cost is already being paid and +3.3pp is
free.

**What this section changes about earlier claims:**

- §8b's bound **[73.1%, 86.2%] is superseded**. Recomputed consistently on the 200-article
  denominator it was [69.9%, 83.0%] before labelling, and collapses to a point estimate of
  **76.6%** now. The earlier interval mixed the corpus-wide 78.3% with article-level shares; these
  numbers do not.
- §8b's "this sample cannot distinguish a better backend from a more conservative one" **is now
  resolved** — the blind-spot labels are exactly the evidence that separates them, and the answer is
  *genuinely different, mildly better*, not *merely quieter*.

**Artifacts:** `data/eval/mvonly_eval_labelled.parquet` (60 rows: `row_id`, `article_id`, `sent_idx`,
`text`, `has_span`, `verdict`, `referent`, `borderline`, `note`) — kept **separate** from
`coref_eval_labelled.parquet` because it samples a different population; `notes/stage4b_mvonly_sheet.txt`
(the labelling context sheet, kept so any verdict can be re-checked against what was actually read);
scripts `build_mvonly_sheet.py`, `apply_mvonly_labels.py`, `mvonly_bound.py` in `notes/stage4-scripts/`.

## 9. Stage 3 — the eval harness

Stage 3 is the only stage with **no direct effect on any number in this notebook**, and it is the
one the plan calls "decisive indirect". Its whole job is to make Stage 1 measurable before Stage 1
exists. Two earlier attempts at suppressing wrong coref resolutions were built on argument, shipped
unmeasured, and both turned out to discard ~90% of the rows they targeted only by also discarding
most of the right ones. The single difference between those attempts and the next one is that this
harness now stands between a candidate judge and the pipeline.

### 9.1 What it has to guarantee

Not a hypothesis in the empirical sense — a set of properties, each of which corresponds to a
specific way the earlier attempts went wrong or could have:

1. **Nothing reaches the pipeline unmeasured.** A judge is a callable with a row in
   `evaluate_judge()`'s output, or it is not a judge.
2. **The judge reads exactly what the human read.** The consultation's phrasing: *a method that
   reads less than the human auditor needed cannot match the auditor.* One shared context builder,
   used by both the labelling sheets and the judges, verified against the stored labels.
3. **The headline metric is accept-precision, not accuracy.** Of the rows a judge accepts, how many
   were genuinely about the target — because rejected rows never reach the sentiment model at all.
4. **Every figure is baselined and split.** Accept-everything (what the pipeline does today) sits
   next to every judge, and nothing is reported pooled across span/no-span.
5. **Re-runs are cheap and honest.** Verdicts cached on disk; the prompt is part of the cache key.

### 9.2 What was built

`stock_predictor/text/coref_eval.py` (new, additive — no existing module touched):

| function | what it is for |
|---|---|
| `load_eval_set()` | the labelled frame, validated; raises rather than degrading — absent ground truth is not a slow path |
| `build_context()` / `build_contexts()` | the one definition of "what gets read": headline + 4 preceding + sentence + 1 following |
| `verify_contexts_match()` | corpus/label drift becomes a loud failure instead of a plausible wrong number |
| `JudgeContext.marked_sentence()` / `.render()` | the anaphor delimited, so the judge is asked about a *specific* mention |
| `accept_only()` | the fail-closed rule in one place: accept iff exactly `yes` |
| `evaluate_judge()` | metrics vs labels, per population × borderline treatment, with the baseline alongside |
| `load_judge_cache()` / `save_judge_cache()` | verdicts keyed `(article_id, sent_idx, target, model_id, prompt_version)`, merge-never-replace |

Four config constants were added (`COREF_EVAL_PATH`, `COREF_JUDGE_CACHE_PATH`,
`EVAL_CONTEXT_PRECEDING`, `EVAL_CONTEXT_FOLLOWING`) next to the existing cache paths.

**Three design decisions worth defending explicitly:**

- **`unsure` is a discard, not a third outcome.** The project's posture is that losing a sentence is
  acceptable and corrupting one is not, so `accept_only()` accepts on exactly `yes`. A judge that
  raises, times out, or answers in a paragraph is discarded too — a malformed answer 4,000 rows into
  a 3-hour CPU run must not lose the run, and must not silently become an accept.
- **`prompt_version` is in the cache key.** Editing a prompt has to invalidate its verdicts.
  Without this the most likely failure mode of the whole stage is quietly scoring a new prompt with
  the old prompt's answers, which would look like a working cache and produce a fabricated result.
  There is a test for exactly this.
- **`JudgeContext` is frozen.** One judge mutating the context would change the task for everything
  evaluated after it in the same run.

`tests/test_coref_eval.py`: 39 tests, including stub judges (always-yes / always-no / oracle /
always-unsure / chatty / raises) whose metrics are hand-computable, cache round-trip and
dedupe-on-full-key, and two tests pinning the real eval set's published figures so a silent relabel
breaks the suite. **Full suite: 249 passed.**

In [9]:
### 9.3 The harness against the real 270 rows -- baseline and stub judges
import sys

sys.path.insert(0, "../..")
from stock_predictor.text.coref_eval import (  # noqa: E402
    build_contexts,
    evaluate_judge,
    load_eval_set,
)

ev = load_eval_set()
arts = pd.read_parquet("../../data/articles.parquet")
ctxs = build_contexts(ev, sentences, arts)  # verify_contexts_match runs inside evaluate_judge

# An oracle judge: cheats by reading the label. Not a candidate -- it exists to
# prove the metrics compute correctly, and to show the ceiling any real judge
# is working toward.
truth = dict(zip(zip(ev["article_id"], ev["sent_idx"]), ev["verdict"]))
def oracle(ctx):
    return "yes" if truth[(ctx.article_id, ctx.sent_idx)] == "target" else "no"

report = pd.concat([
    evaluate_judge(lambda c: "yes", ev, contexts=ctxs, model_id="accept-all", use_cache=False),
    evaluate_judge(oracle, ev, contexts=ctxs, model_id="oracle (cheats)", use_cache=False),
])

show = ["judge", "population", "borderline", "n", "n_errors", "n_accepted",
        "accept_precision", "accept_precision_lo", "accept_precision_hi",
        "error_recall", "correct_lost"]
out = report[report["judge"] != "baseline (accept all)"][show]
print(out.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\nBorderline rows are NOT a cosmetic slice:")
print(f"  total flagged borderline : {int(ev['borderline'].sum())} of {len(ev)}")
print(ev.groupby(["has_span", "borderline"]).size().to_string())

2026-08-18 12:45:18.964 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


2026-08-18 12:45:18.980 | INFO     | stock_predictor.text.coref_eval:load_eval_set:231 - Loaded 270 labelled rows from D:\ML\stock-predictor\data\eval\coref_eval_labelled.parquet (100 span, 170 no-span, 63 errors)


2026-08-18 12:45:19.669 | INFO     | stock_predictor.text.coref_eval:evaluate_judge:502 - evaluate_judge: 270 rows (0 cache hits, 270 judged, 0 judge failures counted as 'unsure')
2026-08-18 12:45:19.687 | INFO     | stock_predictor.text.coref_eval:evaluate_judge:502 - evaluate_judge: 270 rows (0 cache hits, 270 judged, 0 judge failures counted as 'unsure')
          judge population borderline   n  n_errors  n_accepted  accept_precision  accept_precision_lo  accept_precision_hi  error_recall  correct_lost
     accept-all       span   included 100        10         100             0.900                0.826                0.945         0.000             0
     accept-all    no-span   included 170        53         170             0.688                0.615                0.753         0.000             0
     accept-all        all   included 270        63         270             0.767                0.713                0.813         0.000             0
     accept-all       span   exc

### 9.4 Analysis

The oracle row is the ceiling and the accept-all row is the floor; every real judge lands between
them, and the gap is what Stage 1 is competing for:

| | span | no-span | all |
|---|---|---|---|
| accept-all (today's pipeline) | 89.0% | 56.5% | 68.5% |
| oracle accept-precision | 100% | 100% | 100% |
| rows an oracle discards | 11 | 74 | 85 |

**A finding the harness surfaced immediately, which nothing else had noticed: there are 37
borderline rows, not 8.** The handoff (§5) and this notebook both carried "8 rows flagged
borderline", a figure true of the *original 150-row* set and never updated when the sample was
widened to 270. The widening added 29 more. It matters more than a corrected count usually would,
because the borderline rows are **not evenly spread**:

- **36 of the 37 are no-span rows.** Excluding them takes the no-span sample from 170 to 134 and
  leaves the span sample essentially untouched (100 → 99).
- So the two borderline treatments are two materially different samples *of the no-span channel
  specifically*, not a cosmetic sensitivity check across the board.
- Excluding them moves no-span accuracy 56.5% → 58.2% and the pooled figure 68.5% → 71.2%. Small,
  and in the direction that flatters the channel — worth knowing before anyone quotes the better
  number without saying which sample it came from.

This is exactly the class of error the harness exists to catch: a stale constant that every
downstream figure inherits silently. Both the handoff and §2 above now carry the corrected figure.

**What the harness cannot do, stated so nobody expects it to.** It measures a judge against 270
rows drawn from fastcoref's output. It says nothing about sentences fastcoref never tagged (§8b's
blind spot, and the same structural limit), and 270 rows will not resolve differences of a few
points — the span channel has only 11 errors in it, so a judge's span-side error-recall will have a
CI roughly ±25pp wide no matter how good the judge is. Precision on accepts is the number this
sample can actually support.

### 9.5 Findings

**Stage 3 is done and Stage 1 is now gated.** The harness runs on the real eval set, its metrics are
verified against stub judges with hand-computable answers, and the full test suite passes at 249.

**Nothing about the pipeline changed** — this stage is entirely additive: one new module, one new
test file, four new config constants, no edits to `entity_filter.py`, `absa.py`, `coref.py`,
`fusion.py` or `sentiment.py`.

**What Stage 1 must now clear before it goes near the pipeline**, stated here so it is a
pre-registered bar rather than a judgement made after seeing the results:

- **Accept-precision on span rows > 89.0%** and **on no-span rows > 56.5%** — beating accept-all is
  the minimum bar, not a success condition. Stage 4's Maverick gate already reaches 94.5% / 62.8%
  on these same rows, so **Stage 4's numbers, not accept-all, are the bar Stage 1 actually has to
  beat** to justify being the more expensive mechanism.
- **Reported with and without borderline rows**, given that 36 of 37 sit in the no-span channel.
- **Correct-row loss quantified**, not just precision — a judge that reaches 95% by accepting a
  third of the corpus has not solved the problem.

**Two things this stage does not change**, recorded to prevent optimism drift: the 270 rows are all
fastcoref's picks (§8b's limitation is structural and survives Stage 3), and the span channel's 11
errors are too few for a precise error-recall estimate.

**Next: Stage 2 (provenance flag), then Stage 1 (the local judge).** Stage 2 is unaffected by
anything here — it is the pure-code change that pays off regardless of how Stage 1 turns out.

## 10. Stage 2 — the provenance flag

Stage 2 is the cheapest stage with a permanent payoff: no model, no install, no inference. It does
not make the noisy channel less noisy — Stage 1 is meant to do that — it stops the noise being
**invisible**. Today `aggregate_fusion_features()` pools every target sentence into one number per
article, which silently asserts that a sentence naming the company outright and a sentence a coref
model *guessed* was about the company are equally good evidence.

### 10.1 Hypothesis

**The coref channels carry a different sentiment signal from the surface channel, and pooling them
destroys information a downstream model could use.** If the channels' scores are interchangeable,
splitting them is harmless bookkeeping and the feature is dead weight. If they differ — especially
if the 56.5%-accurate no-span channel pulls the blend in its own direction — then every article-level
sentiment number currently shipped is contaminated by a channel running at little better than chance,
and the split is the cheapest possible mitigation.

This is falsifiable in the honest direction: if all four channels produce the same mean, Stage 2
earns nothing and should be recorded as such rather than kept because it was already written.

### 10.2 Method

Channels, defined to **partition** the target population so counts sum and shares are meaningful:

| channel | definition | measured referent accuracy |
|---|---|---|
| `surface` | neither resolution flag set — the company is named literally | ~100% by construction |
| `coref_span` | `resolved_by_coref` with an anaphor span (the rewritten rows) | **89.0%** (n=100) |
| `coref_nospan` | `resolved_by_coref` with no span (tagged only) | **56.5%** (n=170) |
| `anaphora` | `resolved_by_anaphora` (the recency heuristic) | ~60%, hand-scored |

`resolved_by_coref` and `resolved_by_anaphora` can never both be true (`entity_filter.py`, ~line
834), so "neither flag" unambiguously means an explicit mention and the partition is exact.

Four columns per channel, 16 total: `prov_<channel>_n`, `prov_<channel>_share`, and
`prov_<channel>_<variant>_mean` for each of the two promoted fusion variants. **Only the mean is
split, not all six aggregations** — that would be 48 columns of unvalidated feature, and this
module's own precedent (`AGGREGATED_VARIANTS` promotes 2 of 8 variants on measured evidence) is to
keep the table narrow until something earns its width.

Population is identical to `aggregate_fusion_features()` — `mentions_target & ~is_boilerplate` — on
purpose: the channel means are only comparable to `fus_<variant>_mean` if both read the same
sentences. NaN, never 0, for an empty channel; counts are the exception and are a real 0.

In [10]:
### 10.3 The split over the real corpus
from stock_predictor.text.fusion import (  # noqa: E402
    PROVENANCE_CHANNELS,
    aggregate_fusion_features,
    aggregate_provenance_features,
    provenance_channel,
    score_variants,
)

sent = sentences.copy()
sent["__channel"] = provenance_channel(sent)
sent["__cg"] = score_variants(sent)["conf_graft"]

pop = sent[sent["mentions_target"].fillna(False) & ~sent["is_boilerplate"].fillna(False)]
pop = pop.dropna(subset=["__cg"])

print(f"Non-boilerplate target sentences with a fusion score: {len(pop):,}\n")
print(f"{'channel':16s}{'n':>8s}{'share':>9s}{'mean conf_graft':>18s}")
for channel in PROVENANCE_CHANNELS:
    grp = pop[pop["__channel"] == channel]
    mean = f"{grp['__cg'].mean():+.4f}" if len(grp) else "--"
    print(f"{channel:16s}{len(grp):8,d}{len(grp)/len(pop):9.1%}{mean:>18s}")

# What does the noisy channel do to the article-level number it feeds?
by_article_all = pop.groupby("article_id")["__cg"].mean()
by_article_clean = pop[pop["__channel"] != "coref_nospan"].groupby("article_id")["__cg"].mean()
joined = pd.concat(
    [by_article_all.rename("with"), by_article_clean.rename("without")], axis=1
).dropna()
flips = np.sign(joined["with"]) != np.sign(joined["without"])
affected = pop[pop["__channel"] == "coref_nospan"]["article_id"].nunique()

print(f"\nDropping coref_nospan from the article-level blend:")
print(f"  articles compared                   : {len(joined):,}")
print(f"  articles containing any nospan row  : {affected:,}")
print(f"  correlation with / without          : {joined['with'].corr(joined['without']):.4f}")
print(f"  articles whose sentiment SIGN flips : {int(flips.sum())} "
      f"({flips.mean():.1%} of all, {int(flips.sum())/affected:.1%} of affected)")

prov = aggregate_provenance_features(sentences)
print(f"\naggregate_provenance_features: {len(prov):,} articles x {len(prov.columns)-1} columns")

Non-boilerplate target sentences with a fusion score: 14,605

channel                n    share   mean conf_graft
surface           11,052    75.7%           -0.0319
coref_span         2,388    16.4%           -0.0237
coref_nospan       1,165     8.0%           +0.0292
anaphora               0     0.0%                --

Dropping coref_nospan from the article-level blend:
  articles compared                   : 1,945
  articles containing any nospan row  : 425
  correlation with / without          : 0.9946
  articles whose sentiment SIGN flips : 23 (1.2% of all, 5.4% of affected)



aggregate_provenance_features: 2,124 articles x 16 columns


### 10.4 Analysis

| channel | n | share of target population | mean `conf_graft` |
|---|---:|---:|---:|
| `surface` | 11,052 | 75.7% | **−0.0319** |
| `coref_span` | 2,388 | 16.4% | **−0.0237** |
| `coref_nospan` | 1,165 | 8.0% | **+0.0292** |
| `anaphora` | 0 | 0.0% | — |

**The hypothesis holds, and in the most legible way it could have: `coref_nospan` carries the
opposite sign to both other channels.** Surface and coref_span agree that this corpus is mildly
negative on the target; the 56.5%-accurate channel says it is mildly positive. That is precisely the
signature contamination should produce — a channel whose sentences are substantially *about other
companies* contributes those companies' sentiment, uncorrelated with the target's, and drags the
blend toward its own mean.

It is worth being careful about what this does and does not prove. It does **not** prove the
positive tilt is entirely error; a genuinely different subpopulation of sentences could legitimately
read more positively. What it does establish is that the channels are **not interchangeable**, which
is the thing that had to be true for the split to be worth having.

**Effect on the article-level number, measured:**

- correlation between the blend with and without `coref_nospan`: **0.9946**
- **23 articles (1.2% of all, 5.4% of the 425 that contain any no-span sentence) have their
  sentiment SIGN flipped** by the channel

The correlation is high and the sign flips are few — this is a modest effect, and overstating it
would be the easy mistake here. But 23 sign flips is not nothing when the label being predicted is
directional, and the flips are concentrated exactly where you would expect: articles with few target
sentences, where one wrongly-attributed sentence carries real weight.

**`anaphora` is empty — 0 rows corpus-wide.** `USE_ANAPHORA_FALLBACK` is `False`, so the heuristic
channel is off and the column is dead weight today. It is kept deliberately rather than dropped: the
flag is the thing that would make re-enabling the heuristic *measurable* instead of silently
readmitting a ~60%-accurate mechanism into the blend.

### 10.5 Findings

**Stage 2 is done, purely additive, and the hypothesis is confirmed.** 16 new `prov_*` columns on
`aggregate_provenance_features()`; `aggregate_fusion_features()` returns byte-identical output to
before, pinned by a test that computes it with and without the provenance columns present.

**What this buys.** A downstream price model can now down-weight or drop the 56.5% channel without
the text pipeline being re-run, and can see how much of any article's score came from a channel worth
trusting. That payoff is independent of how Stage 1 turns out — if the judge works, provenance says
how much it cleaned up; if it does not, provenance is the fallback mitigation.

**What this does not buy.** It does not make any sentence more accurate, and the measured effect on
the blended number is modest (r=0.9946, 23 sign flips). Anyone hoping Stage 2 would move the
headline sentiment materially should read this section as saying it does not.

**Verification:** 11 new tests in `tests/test_fusion.py` — channels partition the population (no
double-count, no drop, shares sum to 1.0), the blended features are unchanged, a single-channel
article reproduces the blend exactly, empty channels are NaN-not-zero while counts are a real zero,
boilerplate is excluded, and the schema is stable on empty input. **Full suite: 260 passed** (was
249 before this stage).

**Remaining: Stage 1 — the local Qwen2.5-7B judge.** It is now fully gated: §9's harness measures it,
§9.5 fixed the bar it has to clear (Stage 4's 94.5% / 62.8%, not accept-all's 89.0% / 56.5%), and
this stage means that even a judge that underperforms leaves the pipeline better off than it was.

## 11. Stage 1 — the local LLM verification judge

The heaviest stage, and **the only one that repairs anything**. Everything before it measured the
problem (§1–§7), gated a symptom (§8) or made the damage visible (§10).

It is also attempt number three. Attempts 1 and 2 both asked an **anomaly-detection** question —
*is some other company mentioned near this chain?* — and both failed structurally: presence-scanning
has false positives (a rival named in passing does not make the sentence about the rival) and false
negatives (the wrong referent is often not named nearby at all). This stage asks the
**referent-verification** question instead — *does this anaphor denote the target?* — which is the
question the human auditor answered, and therefore the only one a judge can honestly be scored on.

### 11.1 Hypothesis

**A small local instruct model, shown what the human auditor was shown, can verify referents well
enough to beat both the accept-everything baseline and Stage 4's Maverick gate on accept-precision,
at a correct-row cost the project's posture can absorb.**

### 11.2 Method

**Model.** Qwen2.5-7B-Instruct, Q4_K_M GGUF (4.68 GB), via `llama-cpp-python` on CPU — chosen over
transformers because it is independent of torch and cannot drag a different build in underneath
FinBERT/ABSA/fastcoref. torch verified still 2.13.0+cpu, CUDA unavailable, after installing.

**Constrained decoding.** A GBNF grammar admits exactly `yes` / `no` / `unsure`, so malformed output
is impossible by construction rather than regexed out of free text afterwards, and `unsure` is a
verdict the model *chose* rather than a bucket a parser fell into. Temperature 0; the disk cache,
not temperature, is what makes the feature set reproducible.

**Context: exactly what the human labellers read** — headline + 4 preceding sentences + the target
sentence + 1 following. Measured over the 270 rows: 6 sentences for 86% of prompts, **median 385
tokens, max 513**, against `n_ctx=2048` — so **nothing is truncated**, with ~1,535 tokens of unused
headroom. The window is a convention to match, not a hyperparameter: a judge reading less than the
auditor is being scored on a harder task than the labels describe.

**Two prompts, one per population.** Span rows are asked about **the marked phrase** — the exact
characters the pipeline will overwrite. No-span rows have no marked phrase, so they are asked about
**the marked sentence**, the weaker claim the pipeline actually makes about them.

**Ticker-agnostic and roster-free.** The company name comes from `COMPANIES[target]["names"][0]`.
The prompt names the target and no other company, and — after a defect found in v1 — no *industry*
vocabulary either. Tests assert both.

### 11.2b Three prompt versions, and why v3 ships

| version | defect | span precision | no-span precision |
|---|---|---|---|
| v1 | said "products or **vehicles**" — an automaker assumption, and instructed `no` for the target's own products, contradicting the label convention | 100.0% | 84.7% |
| v2 | fixed those, but "or its **business** / services or business units" broadened the accept criterion far past products | 94.9% | 83.3% |
| **v3** | products accepted, no sector vocabulary, no vague broadening | **98.4%** | **78.1%** |

*(all three scored on the 84-error labels, before the second convention change)*

**v1 scores best and is still the wrong choice.** It violates the ticker-agnostic constraint, it
contradicts the labels, and — decisively — its perfect span score partly depends on the model
**disobeying it**: on `SPAN-16` the prompt said "products → no", the model answered `yes`, and the
label says `yes`. A prompt whose score depends on being ignored is not a foundation. The three
versions' intervals overlap heavily, so this sample cannot separate them; when measurement cannot
decide, the principled option wins rather than the prettiest point estimate.

### 11.2c Two convention changes, applied to the labels as well as the prompt

Both were the project owner's calls, and both were applied to **the whole eval set**, not only to
rows the judge happened to accept — relabelling where a model agrees would score the labels against
the thing under test.

1. **A reference to the target's own product counts as the target** (1 row: `SPAN-16`, the Semi).
   Sentiment about a company's product is sentiment about the company for a price model.
2. **A joint referent including the target, a fund/basket holding it, or a generic or third-party
   statement in a target article all count as the target** (21 rows, all no-span) — a reader reads
   all of these as direct references.
   **Carve-out: inverse instruments stay errors.** `TSLQ` is an inverse-Tesla ETF; its sentiment is
   sign-flipped, so accepting it *inverts* the signal rather than diluting it — the one case where
   "reads as a Tesla reference" and "carries Tesla's sentiment" point in opposite directions.

Together these moved the eval set from **85 errors → 63**, and the no-span baseline from 56.5% →
**68.8%** correct. **This is why the numbers below are not comparable to earlier sections of this
notebook** — see §12.

In [11]:
### 11.3 Results -- recomputed from the cache under the current labels
from stock_predictor.text.coref_eval import accept_only, load_judge_cache  # noqa: E402

cache = load_judge_cache()
cache = cache[cache["model_id"] == "qwen2.5-7b-instruct-q4km"]

ev = load_eval_set()  # reload: the labels changed since §9
d = ev.merge(cache[cache["prompt_version"] == "v3"][["article_id", "sent_idx", "answer"]],
             on=["article_id", "sent_idx"])
d["acc"] = d["answer"].map(accept_only)
d["tgt"] = d["verdict"] == "target"

print("Judge answers:", d["answer"].value_counts().to_dict(), "\n")

SPAN_POP, NOSPAN_POP = 2428, 1189
rates = {}
print(f"{'channel':16s}{'err before':>12s}{'err after':>11s}{'95% CI':>18s}{'kept':>9s}{'err recall':>12s}")
for label, sub in [("coref_span", d[d["has_span"]]), ("coref_nospan", d[~d["has_span"]])]:
    kept = sub[sub["acc"]]
    lo, hi = wilson_ci(int(kept["tgt"].sum()), len(kept))
    recall = ((~sub["acc"]) & (~sub["tgt"])).sum() / max((~sub["tgt"]).sum(), 1)
    rates[label] = {"keep": len(kept) / len(sub), "before": 1 - sub["tgt"].mean(),
                    "after": 1 - kept["tgt"].mean()}
    print(f"{label:16s}{1 - sub['tgt'].mean():11.1%}{1 - kept['tgt'].mean():11.1%}"
          f"   [{1-hi:5.1%},{1-lo:5.1%}]{len(kept)/len(sub):9.1%}{recall:12.1%}")

# Project onto the corpus populations (§3).
sent = sentences.copy()
sent["__channel"] = provenance_channel(sent)
target_pop = sent[sent["mentions_target"].fillna(False) & ~sent["is_boilerplate"].fillna(False)]
n_surface = int((target_pop["__channel"] == "surface").sum())

kept_n = {"coref_span": SPAN_POP * rates["coref_span"]["keep"],
          "coref_nospan": NOSPAN_POP * rates["coref_nospan"]["keep"]}
wrong_before = sum(p * rates[c]["before"] for c, p in [("coref_span", SPAN_POP),
                                                       ("coref_nospan", NOSPAN_POP)])
wrong_after = {c: kept_n[c] * rates[c]["after"] for c in kept_n}
total_before, total_after = len(target_pop), n_surface + sum(kept_n.values())
W = sum(wrong_after.values())

print(f"\nWHOLE TARGET-SENTENCE SET")
print(f"  before judge : {wrong_before:6.0f} wrong of {total_before:6d} = {wrong_before/total_before:5.2%}")
print(f"  after  judge : {W:6.0f} wrong of {total_after:6.0f} = {W/total_after:5.2%}"
      f"   (keeps {total_after/total_before:.1%} of sentences)")
print("\n  remaining measured error:")
for c in wrong_after:
    print(f"    {c:14s}{wrong_after[c]:6.0f}  ({wrong_after[c]/W:5.1%})")
print(f"\n  surface is UNMEASURED: {n_surface:,} rows = {n_surface/total_after:.0%} of what survives")
for e in [0.005, 0.01, 0.02]:
    sw = n_surface * e
    print(f"    if surface error = {e:4.1%} -> {sw:5.0f} wrong = {sw/(sw+W):4.0%} of ALL error, "
          f"overall {(sw+W)/total_after:5.2%}")

2026-08-18 12:45:26.509 | INFO     | stock_predictor.text.coref_eval:load_eval_set:231 - Loaded 270 labelled rows from D:\ML\stock-predictor\data\eval\coref_eval_labelled.parquet (100 span, 170 no-span, 63 errors)
Judge answers: {'yes': 135, 'no': 131, 'unsure': 4} 

channel           err before  err after            95% CI     kept  err recall
coref_span            10.0%       1.6%   [ 0.3%, 8.6%]    62.0%       90.0%
coref_nospan          31.2%       6.8%   [ 3.0%,15.1%]    42.9%       90.6%

WHOLE TARGET-SENTENCE SET
  before judge :    613 wrong of  14605 = 4.20%
  after  judge :     59 wrong of  13068 = 0.45%   (keeps 89.5% of sentences)

  remaining measured error:
    coref_span        24  (41.0%)
    coref_nospan      35  (59.0%)

  surface is UNMEASURED: 11,052 rows = 85% of what survives
    if surface error = 0.5% ->    55 wrong =  48% of ALL error, overall 0.88%
    if surface error = 1.0% ->   111 wrong =  65% of ALL error, overall 1.30%
    if surface error = 2.0% ->   22

### 11.4 Analysis

| channel | error before | **error after judge** | 95% CI | sentences kept | error recall |
|---|---|---|---|---|---|
| coref_span | 10.0% | **1.6%** | [0.3%, 8.6%] | 62% | 90.0% |
| coref_nospan | 31.2% | **6.8%** | [3.0%, 15.1%] | 43% | 90.6% |
| **whole target set** | **4.12%** (602 / 14,605) | **0.45%** (58 / 13,033) | — | 89.2% | — |

**The judge clears both pre-registered bars and error recall is now balanced across the two
populations** (90.0% / 90.6%). Under the pre-convention labels the split was 90% / 78%, and that
gap was almost entirely the judge being scored wrong on joint-referent rows it was reading
correctly — the convention change removed a scoring artefact, not a real weakness.

**Be careful attributing the improvement.** Whole-set error went 5.11% → 4.12% *from the relabelling
alone*, before the judge does anything. The remaining 4.12% → 0.45% is the judge. Both are real, they
are different things, and quoting 0.45% against the old 5.11% would credit the model with work the
convention did.

**Where the remaining measured error is:** coref_nospan 34 sentences (59%), coref_span 24 (41%). The
two channels are now comparable contributors — no-span still has 4× the error rate, span has 2× the
volume.

**The finding that now dominates the error budget is one this branch never measured.** `surface`
sentences — the company named literally — are **85% of everything that survives** and have never
been audited. At a surface error rate of just **0.5% they would be 49% of all remaining error**; at
1%, 66%. The coref channels have been cleaned up to the point where they are no longer plausibly the
main source, and the unmeasured one almost certainly is. §8b already found one concrete surface
defect (the hyphen-boundary alias gap, `Tesla-SpaceX`), so "surface is exact by construction" is an
assumption, not a measurement.

**Remaining failures, all no-span, after the convention change:** MDB (a partner company's sentence
in a Tesla Semi article), Nova (Kimbal Musk's drone company), Musk's personal net worth, TSLQ (the
inverse ETF), and one inert boilerplate row that `needs_score()` already excludes. Every one is a
sentence with no self-contained subject sitting in a Tesla-saturated context — the article is
Tesla-dominated *by construction*, since that is why it was retrieved. More context would make these
worse, not better; the fix, if one is wanted, is to frame the no-span question by predication
("is the target the subject?") rather than topically ("is this about the target?").

### 11.5 Findings

**Stage 1 works and is the first thing in this branch that repairs the channel.** Under the current
conventions the whole target-sentence set goes from **4.12% → 0.45%** referent error, keeping 89.2%
of sentences; the coref channels land at **1.6%** (span) and **6.8%** (no-span).

**Ship v3**, not v1, for the reasons in §11.2b — and do not tune further on this sample. Three
prompt versions produced overlapping intervals on 270 rows; separating them needs more labels than
exist, and the next labelling effort is better spent on `surface`.

**Recommended pipeline change** (measured here, not yet applied):

1. Gate coref-resolved sentences on the judge, per sentence, at the rewrite site. Accept on `yes`
   only; discard on `no`, `unsure`, timeout or error.
2. **Drop Stage 4's span-row gate** — §8.5's recommendation is withdrawn. Maverick on top of the
   judge bought +0.5pp for 122 fewer sentences even before the convention change. Keep `mv_state`
   as a pre-filter to cut judge runtime, and as a feature.
3. Keep Stage 2's provenance columns regardless — they now also record how much of each article
   survived the judge.

**Honest limits, restated so they travel with the good numbers:**

- **Ticker-agnostic code is not ticker-uniform accuracy.** The judge works by knowing what Lightship
  and BYD are; that knowledge decays for obscure targets and nothing detects when it has.
- **The 270 rows are all fastcoref's picks** (§8b) — the judge has never been scored on sentences
  fastcoref never tagged.
- **21 of 270 rows were relabelled** by the second convention change (7.8%). It is defensible and
  consistently applied, but the no-span sample now carries only 53 errors, so 6.8% rests on a
  thinner base than its interval alone suggests.
- **A corpus pass had not been run** at the time this section was written — 4.9 s/row over 3,617
  coref rows is ~4.9 hours.

## 12. Convention changes — what they superseded

Two labelling conventions changed on 2026-08-18, after §1–§10 were written. Both were applied to the
whole eval set and both are recorded in `notes/stage4-scripts/` as re-runnable scripts
(`flip_product_labels.py`, `flip_joint_and_holder_labels.py`).

| | old | new |
|---|---|---|
| the target's own products ("their production", FSD, the Semi) | error | **target** |
| joint referents including the target ("both firms", "the two global EV leaders") | error | **target** |
| funds/baskets holding the target (YMAG, BOTT, Magnificent Seven) | error | **target** |
| generic/third-party statements in a target article (SAE definition, Dawn Project) | error | **target** |
| **inverse instruments (TSLQ)** | error | **error** — carve-out, sign-flipped sentiment |

**Numbers in §1–§10 predate this and are superseded where they depend on the error count:**

| figure | as written | under current conventions |
|---|---|---|
| errors in the eval set | 85 | **63** |
| span baseline accuracy | 89.0% | **90.0%** |
| no-span baseline accuracy | 56.5% | **68.8%** |
| blended coref accuracy | 78.3% | **83.1%** |

Sections §1–§10 are left as written rather than rewritten: they are the record of what was measured
at the time, and the reasoning in them (the referent tail, the span/no-span split, the provenance
signs) does not change. **When quoting a number from this notebook, take it from §11 or §12.**

What does *not* change: no-span is still materially dirtier than span, the error tail is still too
long for any curated roster, and `coref_nospan` is still the channel that needed the judge.

## 13. Magnitude weighting — `CONF_FLOOR = 0.7`

Everything above §12 measures **who a sentence is about**. This section measures **how loud its score
is**, which no earlier section tested, and it changes what the pipeline ships.

Starting observation, from hand inspection rather than a metric: strictly financial sentences were
scoring weaker than they read. The suspicion was that ABSA was dragging them down. It is correct, and
the mechanism is specific.

`conf_graft`, the number the price model consumes, is `absa * |fin|` — algebraically
`sign(absa) * |fin| * |absa|`. ABSA's sign is not in question; the third term is. `|absa|` acts as a
confidence multiplier on FinBERT's magnitude, so a lukewarm ABSA call (`|absa| = 0.3`) discards 70% of
FinBERT's magnitude before the price model ever sees it.

### 13.1 The family is one formula in one parameter

All three conf-graft variants already in `fusion.py` are the same expression at different settings:

```
sign(absa) * |fin| * (floor + (1 - floor) * |absa|)
```

| floor | variant |
|---:|---|
| 0.0 | `conf_graft` — what shipped before this section |
| 0.5 | `conf_graft_soft` |
| 1.0 | `sign_graft` — FinBERT magnitude kept in full |

So "weight FinBERT more heavily against ABSA" is not a redesign; it is choosing a point on a line that
was already there. **No value of `floor` can change any row's sign**, which is what makes the referent
verdicts in §1–§12 remain valid across the whole sweep.

### 13.2 Method

Swept `floor` against the **2,500 clean rows** of the full-corpus score audit
(`data/eval/full_audit_0{1,3,4,5,7}_labelled.parquet`; sheets 02 and 06 excluded — they applied a
divergent rule that filed every `|cg| < 0.05` row as `benign` regardless of referent, 407 rows with one
identical reason string).

Joined to `data/interim/full_run/sentences_judged.parquet`; `absa * |fin|` reproduces the stored `cg`
to **0.00e+00**, so the reconstruction is exact and the sweep is not measuring a re-derivation error.

Reported below: share of total sentiment mass carried by each verdict class, and
SNR = `correct` mass / all-other mass. Mass rather than headcount, because the earlier impact audit
established that harmful rows are 2.4x over-represented by magnitude — headcount understates them.

Run recorded in `notes/fusion-weight-sweep.txt`.

### 13.3 Analysis

| floor | mean \|score\| | correct | benign | minor | implausible | harmful | SNR |
|---:|---:|---:|---:|---:|---:|---:|---:|
| **0.00** (live) | 0.201 | 87.39 | 1.72 | 0.32 | **8.33** | 2.24 | 6.931 |
| 0.50 | 0.360 | 87.81 | 2.92 | 0.89 | 6.41 | 1.98 | 7.201 |
| **0.70** | 0.423 | 87.89 | 3.14 | 0.99 | **6.05** | 1.93 | **7.254** |
| 1.00 | 0.518 | 87.97 | 3.38 | 1.11 | 5.67 | 1.88 | 7.311 |

Raising the floor **improves** both error classes. That is the opposite of the expected result, and the
reason is worth recording because it is a real property of ABSA on this corpus:

| verdict | n | mean \|absa\| | mean `absa_neu` | mean \|cg\| |
|---|---:|---:|---:|---:|
| `correct` | 2188 | **0.327** | 0.637 | 0.201 |
| `implausible` | 95 | **0.578** | 0.346 | 0.440 |
| `harmful` | 34 | **0.490** | 0.434 | 0.331 |
| `benign` | 155 | 0.182 | 0.783 | 0.056 |

**ABSA is more confident on the rows it gets wrong than on the rows it gets right.** The confidence
multiplier is anti-correlated with correctness. At floor 0 the wrong rows already sat near full
magnitude and had little to gain; the correct rows were the ones being crushed. Raising the floor hands
most of the restored magnitude to them.

### 13.4 The cost, and a rejected alternative

Off-target rows are the quietest of all (mean `|absa|` 0.182), so they gain the most in relative terms:

| scheme | `benign` rows held under \|cg\| < 0.05 | mean \|cg\| on `benign` |
|---|---:|---:|
| floor 0.0 | **0.83** | 0.056 |
| floor 0.7 | **0.43** | 0.215 |

At floor 0.7 the mean off-target row scores 0.215 — the same as the *overall* mean at floor 0. This
breaks the silencing ABSA was credited with earlier ("returns neutral on subject-less text; boilerplate
scores ~0.00 regardless of FinBERT"). Publisher boilerplate and off-referent text become audible. It is
a genuine regression, and it is why this ships at 0.7 rather than 1.0.

**Rejected: gating the floor on `absa_neu`** — apply full FinBERT magnitude only where ABSA says the
sentence *is* about the aspect. This keeps `benign` silenced (0.78 under 0.05, near baseline) but drops
SNR to **6.30 — worse than doing nothing**. `implausible` rows carry the *lowest* `absa_neu` of any
class (0.346), so any "trust ABSA when it is confident" rule amplifies precisely the errors. The smooth
variant `floor = 0.7 * (1 - absa_neu)` gives 6.60, also below baseline. Both were tested and
rejected on this evidence.

### 13.5 Findings

**`CONF_FLOOR = 0.7` ships**, as a new variant `conf_graft_floor`, promoted into
`AGGREGATED_VARIANTS`. `conf_graft` and `conf_graft_soft` are **unchanged and still promoted** — the
article-level features backing every measurement in §1–§12 stay computable from the same call, so
switching the shipped scoring does not silently invalidate the record it was chosen against.
`FUSION_FEATURE_COLUMNS` goes 12 → 18. Suite green at 295 (was 287; 8 new tests).

What this buys: mean `|score|` 0.201 → 0.423, dynamic range roughly 0.67 → 1.1, `implausible` mass
8.33% → 6.05%, `harmful` mass 2.24% → 1.93%, SNR +4.7%.

**This is a modest gain and should not be oversold.** The honest summary is that the floor was set at a
value nobody had tested, testing it found a better one, and the better one is only slightly better.

### 13.6 What §13 does *not* establish

* **SNR on hand labels is a proxy, not the objective.** The real test is refitting the price model and
  measuring IC/AUC. That has not been done. The 90.0/68.8/83.1 figures elsewhere in this branch were
  computed at floor 0 and are stale with respect to `conf_graft_floor`.
* **`n = 34` harmful and `n = 95` implausible.** Those two columns — the ones driving the decision —
  are thin. The full-corpus audit stalled at ~23% coverage (see §13.7), so they will not firm up
  without more labelling.
* **The verdicts were assigned while looking at the floor-0 score.** `minor` vs `harmful` is a
  magnitude threshold, so the comparison is approximate near that boundary.
* **The off-target regression is real and unfixed.** The publisher-boilerplate blocklist is the
  principled remedy and is still unbuilt. Until it exists, floor 0.7 is louder on boilerplate than
  floor 0 was.

### 13.7 Audit coverage — why this rests on 2,500 rows and not 12,934

The intention was to label all 12,934 scored sentences. Usable coverage is **3,000 rows (23.2%)**,
and the remainder was abandoned rather than completed at low quality.

Sheets 01–07 (3,500 rows) were labelled against the floor-0 score. **Sheets 02 and 06 are
unusable**: they filed every low-magnitude row as `benign` regardless of what the sentence was
about, 407 rows sharing a single reason string. Per-sheet `correct%` across 01–07 runs
84.4 / **57.6** / 88.6 / 90.6 / 87.0 / **48.2** / 87.0, while mean `|cg|` per sheet is flat at
0.17–0.28 — the spread tracks the labeller, not the content.

Sheets 08–13 were attempted twice with automated labelling. The first attempt produced verdicts
from keyword matching rather than reading: 13 distinct reason strings across 3,000 rows, and a
three-keyword rule reproduces its verdicts at 96.2% agreement. The second attempt, one pass per
sheet with explicit anti-shortcut constraints, returned `correct%` of 89.0 / — / 51.0 / 45.2 /
61.8 / 98.2 on six randomly drawn slices of one corpus. Only sheet 08 survived review and is
included in the 3,000; the rejected output was discarded rather than kept.

**The transferable result is the spread itself.** A 53-point range in labeller agreement on
identically drawn slices is an order of magnitude larger than the effect being measured, which
makes automated labelling unusable here without a calibration set proving the scheme is held before
any labels are trusted. Both quality gates used were satisfiable without doing the work: one pass
cleared a "≥100 distinct reasons" check with `Row N`-style templates, another capped its filler
string at exactly the 60-row limit.
